# X-ray $\log N - \log S$ distribution

In this notebook, we compare the $\log N - \log S$ of the simulated X-ray population from the best-inferred parameters with the observed population and produce Figures 4 and 8 of Ronchi et al. (2026).

Note that in order to make this notebook work, you first need to download the results data from `/data/magnesia/common/paper_ronchi_etal_2025/B_double_lognormal_dip-tor_heavy/experiments_paper.zip`, copy it into the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026` and unpack the file so that the results data will be saved in the folder `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper`.

In [ ]:
# import libraries
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pathlib
import os

import mlpoppyns.simulator.basics.constants as const
from mlpoppyns.simulator.config_simulator import cfg
import utilities.plot_settings

from utilities.load_catalogs import (
    load_atnf_meerkat_catalog,
    load_xray_catalog,
)

from matplotlib.ticker import FuncFormatter

formatter = FuncFormatter(lambda y, _: "{:.16g}".format(y))
from matplotlib.ticker import LogFormatterMathtext

from scipy.stats import gaussian_kde
from scipy.interpolate import interp1d
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde, ks_2samp

In [ ]:
def tage_from_timing(P: float, Pdot: float) -> float:
    """
    Pulsar characteristic age from timing properties assuming P >> P0 and constant magnetic field.

    Args:
        P (float): Spin period of a simulated pulsar, measured in [s].
        Pdot (float): Period derivative of a simulated pulsar in [s/s].

    Returns:
        (float): Characteristic age in [yr].
    """

    # Characteristic age definition.
    tage = P / (2 * Pdot) / const.YR_TO_S

    return tage

## Load the observed catalogs

In [ ]:
surveys_xray = load_xray_catalog(
    "../../data/observations/thermal_NS_05-11-2024.csv",
)

In [ ]:
P_x_obs = surveys_xray["P"]
Pdot_x_obs = surveys_xray["P_dot"]
S_x_obs = surveys_xray["S_x_abs"]
age_x_real_obs = surveys_xray["age"]
d_x_real_obs = surveys_xray["dist"]

# Compute the characteristic age in kyr.
age_char_x_obs = tage_from_timing(P_x_obs, Pdot_x_obs) / 1000

# Define the filters for the observed young magnetars and XDINSs.
young_obs_mask = (age_x_real_obs <= 2) | (age_char_x_obs <= 2)
xdins_obs_mask = ((age_x_real_obs >= 5) | (age_char_x_obs >= 5)) & (
    d_x_real_obs <= 0.5
)

young_xdins_obs_mask = young_obs_mask | xdins_obs_mask

In [ ]:
S_x_young_xdins_obs = S_x_obs[young_xdins_obs_mask]

In [ ]:
# Compute the logN-logS for the observations.
# Sort the flux densities in ascending order.
sorted_S_x_obs = np.sort(S_x_obs)

# Calculate the cumulative number of sources.
cumulative_number_obs = np.arange(len(sorted_S_x_obs), 0, -1)

# Compute the logN-logS for the observations considering only young magnetars and XDINSs.
sorted_S_x_young_xdins_obs = np.sort(S_x_young_xdins_obs)

# Calculate the cumulative number of sources.
cumulative_number_young_xdins_obs = np.arange(len(sorted_S_x_young_xdins_obs), 0, -1)

## Load the simulations

We have simulated 100 populations of neutron stars using the best-parameter values sampled from the inferred posterior distribution. 
- When the entire observed X-ray population is considered (for Figure 4), we use the trained posterior estimator saved at the following path: `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper/tsnpe_experiment_1_maps8_res32/learning/models/SBI_ConvolutionMDN/20260115_142235/round_4`.
- When only the sample of young magnetars and XDINSs is considered for inference (for Figure 8), we use the trained posterior estimator saved at the following path: `ML-Poppyns/data/paper_results/ronchi_etal_2026/experiments_paper/tsnpe_experiment_1_maps8_res32_youngxdins/learning/models/SBI_ConvolutionMDN/20260202_122351/round_4`.

In [ ]:
root_path = "../../data/paper_results/ronchi_etal_2026/experiments_paper"

# By default, we consider the entire X-ray sample to reproduce Figures 3 and 4.
# To consider the inference results using only young magnetars and XDINSs and
# reproduce Figures 7 and 8, we have to set `use_young_xdins_only` to True.
use_young_xdins_only = True

if use_young_xdins_only:
    simulations_path = f"{root_path}/tsnpe_experiment_1_maps8_res32_youngxdins/simulations_best_params/output_simulations"
else:
    simulations_path = (
        f"{root_path}/tsnpe_experiment_1_maps8_res32/simulations_best_params/output_simulations"
    )

# Number of samples in the parsed directory.
n_sim = len(next(os.walk(simulations_path))[1])

print(n_sim)

In [ ]:
S_x_sim_sorted_list = []
logNlogS_x_sim_list = []

S_x_young_xdins_sim_sorted_list = []
logNlogS_x_young_xdins_sim_list = []

In [ ]:
# Load simulation data and save the relevant parameters into lists. In particular,
# we need the X-ray fluxes and the corresonding logN-logS distributions.

for i in range(n_sim):
    path_to_simulation = f"{simulations_path}/{i:06d}"

    config_json = json.load(
        open(
            pathlib.Path().joinpath(path_to_simulation, "configuration.json"),
        )
    )

    # Skip simulations where the birth rate exceeded the maximum allowed to not bias the birth rate estimate.
    if (
        (config_json["birth_rate_PMPS_at_match"] == 0)
        | (config_json["birth_rate_SMPS_at_match"] == 0)
        | (config_json["birth_rate_HTRU_low_mid_at_match"] == 0)
        | (config_json["birth_rate_xray_realistic_at_match"] == 0)
    ):
        continue

    # Load the `.pkl.gz` files containing the survey results to import.
    df_x_sim = pd.read_pickle(
        pathlib.Path().joinpath(
            path_to_simulation, "survey_xray_realistic_results.pkl.gz"
        ),
        compression="gzip",
    )

    # Extract relevant quantities.
    S_x_sim = df_x_sim["S_x_rcs_abs"]["[erg s^-1 cm^-2]"].to_numpy()
    d_x_sim = df_x_sim["dist"]["[kpc]"].to_numpy()
    age_x_sim = df_x_sim["age"]["[yr]"].to_numpy()

    # Filter young magnetars.
    young_sim_mask = age_x_sim <= 2.0e3
    
    # Filter X-ray emitting NSs with XDINS-like properties.
    xdins_sim_mask = (d_x_sim <= 0.5) & (age_x_sim >= 1.0e5)

    young_xdins_sim_mask = young_sim_mask | xdins_sim_mask

    S_x_young_xdins_sim = S_x_sim[young_xdins_sim_mask]

    # Compute the logN-logS.
    # Sort the flux densities in ascending order.
    sorted_S_x_sim = np.sort(S_x_sim)
    sorted_S_x_young_xdins_sim = np.sort(S_x_young_xdins_sim)

    # Calculate the cumulative number of sources.
    cumulative_number_sim = np.arange(len(sorted_S_x_sim), 0, -1)
    cumulative_number_young_xdins_sim = np.arange(len(sorted_S_x_young_xdins_sim), 0, -1)

    # Append the values to the corresponding lists.
    S_x_sim_sorted_list.append(sorted_S_x_sim)
    logNlogS_x_sim_list.append(cumulative_number_sim)

    S_x_young_xdins_sim_sorted_list.append(sorted_S_x_young_xdins_sim)
    logNlogS_x_young_xdins_sim_list.append(cumulative_number_young_xdins_sim)

Compute the $\log N-\log S$ considering all simulated neutron stars.

In [ ]:
# Interpolate the simulated logN-logS on a common grid to later compute some statistics.
xmin = min(x.min() for x in S_x_sim_sorted_list)
xmax = max(x.max() for x in S_x_sim_sorted_list)

# Define a common grid (e.g. 300 points).
x_common = np.logspace(np.log10(xmin), np.log10(xmax), 1000)

y_interp = []

for x, y in zip(S_x_sim_sorted_list, logNlogS_x_sim_list):
    f = interp1d(x, y, bounds_error=False, fill_value=(np.nan, 0.0))
    y_interp.append(f(x_common))

# Stack values according to shape (N_curves, N_common).
y_interp = np.vstack(y_interp)

In [ ]:
# Compute the percentiles for the median LogN-logS and the 1-sigma and 3-sigma uncertainties.
p01 = np.nanpercentile(y_interp, 0.15, axis=0)
p16 = np.nanpercentile(y_interp, 15.85, axis=0)
p50 = np.nanpercentile(y_interp, 50, axis=0)  # median
p84 = np.nanpercentile(y_interp, 84.15, axis=0)
p99 = np.nanpercentile(y_interp, 99.85, axis=0)

Compute the $\log N-\log S$ considering only simulated young magnetars and XDINSs samples.

In [ ]:
# Interpolate the simulated logN-logS on a common grid to later compute some statistics.
xmin = min(x.min() for x in S_x_young_xdins_sim_sorted_list)
xmax = max(x.max() for x in S_x_young_xdins_sim_sorted_list)

# Define a common grid (e.g. 300 points).
x_common_young_xdins = np.logspace(np.log10(xmin), np.log10(xmax), 1000)

y_interp = []

for x, y in zip(S_x_young_xdins_sim_sorted_list, logNlogS_x_young_xdins_sim_list):
    f = interp1d(x, y, bounds_error=False, fill_value=(np.nan, 0.0))
    y_interp.append(f(x_common))

# Stack values according to shape (N_curves, N_common).
y_interp = np.vstack(y_interp)

In [ ]:
# Compute the percentiles for the median LogN-logS and the 1-sigma and 3-sigma uncertainties.
p01_young_xdins = np.nanpercentile(y_interp, 0.15, axis=0)
p16_young_xdins = np.nanpercentile(y_interp, 15.85, axis=0)
p50_young_xdins = np.nanpercentile(y_interp, 50, axis=0)  # median
p84_young_xdins = np.nanpercentile(y_interp, 84.15, axis=0)
p99_young_xdins = np.nanpercentile(y_interp, 99.85, axis=0)

## Plot the X-ray $\log N - \log S$ distribution

Compare the $\log N - \log S$ distribution of the simulated neutron star population with the observations.
If `use_young_xdins_only = False`, this reproduces Figure 4 in Ronchi et al. (2026).
If `use_young_xdins_only = True`, this reproduces Figure 8 in Ronchi et al. (2026).

In [ ]:
# Create the logN-logS plot.
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    x_common, p50, label="Simulated median", linewidth=4, color="tab:purple"
)

ax.fill_between(
    x_common,
    p16,
    p84,
    alpha=0.4,
    label=r"Simulated $1\sigma$",
    color="tab:purple",
)
ax.fill_between(
    x_common,
    p01,
    p99,
    alpha=0.2,
    label=r"Simulated $3\sigma$",
    color="tab:purple",
)


ax.plot(
    sorted_S_x_obs,
    cumulative_number_obs,
    lw=4,
    color="black",
    label="Observed",
)

# Set logarithmic scale.
ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1.0e-15, 1.0e-9)
ax.set_ylim(0.5, 1.0e3)

# Adding titles and labels.
plt.xlabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
plt.ylabel(r"$N(>S_{X, \rm{abs}})$")

# Add grid lines.
# plt.grid(True, which="both", ls="--")
plt.grid()

ax.legend(frameon=True, loc=0, prop={"size": 25})

if use_young_xdins_only:
    fig.savefig("plots/Figure_8_right.pdf", bbox_inches="tight")
else:
    fig.savefig("plots/Figure_4.pdf", bbox_inches="tight")

# Show the plot.
plt.show()

In [ ]:
# Create the logN-logS plot.
fig, ax = plt.subplots(figsize=(12, 8))

ax.plot(
    x_common_young_xdins, p50_young_xdins, label="Simulated median", linewidth=4, color="tab:purple"
)

ax.fill_between(
    x_common_young_xdins,
    p16_young_xdins,
    p84_young_xdins,
    alpha=0.4,
    label=r"Simulated $1\sigma$",
    color="tab:purple",
)
ax.fill_between(
    x_common_young_xdins,
    p01_young_xdins,
    p99_young_xdins,
    alpha=0.2,
    label=r"Simulated $3\sigma$",
    color="tab:purple",
)


ax.plot(
    sorted_S_x_young_xdins_obs,
    cumulative_number_young_xdins_obs,
    lw=4,
    color="black",
    label="Observed",
)

# Set logarithmic scale.
ax.set_xscale("log")
ax.set_yscale("log")

ax.set_xlim(1.0e-15, 1.0e-9)
ax.set_ylim(0.5, 1.0e3)

# Adding titles and labels.
plt.xlabel(r"$S_{X, \rm{abs}}$ [erg s$^{-1}$ cm$^{-2}$]")
plt.ylabel(r"$N(>S_{X, \rm{abs}})$")

# Add grid lines.
# plt.grid(True, which="both", ls="--")
plt.grid()

ax.legend(frameon=True, loc=0, prop={"size": 25})

if use_young_xdins_only:
    fig.savefig("plots/Figure_8_left.pdf", bbox_inches="tight")

# Show the plot.
plt.show()